# 03 - Train Models

In this notebook, we'll:
1. Train Enhanced XGBoost with all features
2. Train Deep Neural Network with embeddings
3. Evaluate both models
4. Compare with baselines
5. Analyze feature importance

In [ ]:
# Imports
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import pickle

from src.items import Item
from src.features import FeatureEngineer
from src.models import XGBoostPricer, NeuralNetPricer
from src.evaluator import evaluate, compare_models

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Step 1: Load Data

In [ ]:
# Load RAG-augmented items
print("Loading RAG-augmented items...")
with open('../data/train_items_rag.pkl', 'rb') as f:
    train = pickle.load(f)

with open('../data/val_items_rag.pkl', 'rb') as f:
    val = pickle.load(f)

with open('../data/test_items_rag.pkl', 'rb') as f:
    test = pickle.load(f)

print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test items")

In [ ]:
# Load embeddings
print("Loading embeddings...")
train_embeddings = np.load('../data/embeddings/train_embeddings.npy')
val_embeddings = np.load('../data/embeddings/val_embeddings.npy')
test_embeddings = np.load('../data/embeddings/test_embeddings.npy')

print(f"Train embeddings: {train_embeddings.shape}")
print(f"Val embeddings: {val_embeddings.shape}")
print(f"Test embeddings: {test_embeddings.shape}")

In [ ]:
# Extract prices
train_prices = np.array([item.price for item in train])
val_prices = np.array([item.price for item in val])
test_prices = np.array([item.price for item in test])

print(f"Price ranges:")
print(f"Train: ${train_prices.min():.2f} - ${train_prices.max():.2f}")
print(f"Val: ${val_prices.min():.2f} - ${val_prices.max():.2f}")
print(f"Test: ${test_prices.min():.2f} - ${test_prices.max():.2f}")

## Step 2: Prepare Features with RAG

In [ ]:
# Extract enhanced features including RAG
def extract_features_with_rag(items):
    features = []
    for item in items:
        feat = item.get_feature_dict()
        # Add RAG features
        feat['rag_mean_price'] = item.rag_mean_price
        feat['rag_median_price'] = item.rag_median_price
        feat['rag_std_price'] = item.rag_std_price
        feat['rag_weighted_price'] = item.rag_weighted_price
        features.append(feat)
    return pd.DataFrame(features)

train_features = extract_features_with_rag(train)
val_features = extract_features_with_rag(val)
test_features = extract_features_with_rag(test)

print(f"Feature shape: {train_features.shape}")
print(f"\nFeatures: {list(train_features.columns)}")

In [ ]:
# Display feature statistics
train_features.describe()

## Step 3: Train Enhanced XGBoost

In [ ]:
# Initialize XGBoost model
print("Training Enhanced XGBoost...\n")

xgb_model = XGBoostPricer(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
)

# Train with early stopping
xgb_model.fit(
    train_features, 
    train_prices,
    eval_set=(val_features, val_prices),
    early_stopping_rounds=50,
    verbose=True
)

In [ ]:
# Feature importance
importance_df = xgb_model.get_feature_importance(top_n=15)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='skyblue')
plt.xlabel('Importance')
plt.title('Top 15 Feature Importance - XGBoost')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Features:")
print(importance_df.head(10))

In [ ]:
# Evaluate XGBoost
print("\nEvaluating XGBoost on test set...")
xgb_predictions = xgb_model.predict(test_features)

from src.evaluator import compute_metrics
xgb_metrics = compute_metrics(xgb_predictions, test_prices)

print("\n" + "="*60)
print("ENHANCED XGBOOST RESULTS")
print("="*60)
print(f"MAE: ${xgb_metrics['mae']:.2f}")
print(f"RMSE: ${xgb_metrics['rmse']:.2f}")
print(f"MAPE: {xgb_metrics['mape']:.2f}%")
print(f"R²: {xgb_metrics['r2']:.4f}")
print(f"Within 10%: {xgb_metrics['within_10_pct']:.1f}%")
print(f"Within 20%: {xgb_metrics['within_20_pct']:.1f}%")
print("="*60)

In [ ]:
# Save XGBoost model
import os
os.makedirs('../data/models', exist_ok=True)
xgb_model.save('../data/models/xgboost_model.pkl')

## Step 4: Train Deep Neural Network

In [ ]:
# Initialize Neural Network
print("Training Deep Neural Network...\n")

nn_model = NeuralNetPricer(
    embedding_dim=384,  # all-MiniLM-L6-v2 dimension
    feature_dim=train_features.shape[1],
    hidden_dims=[256, 128, 64, 32],
    learning_rate=0.001
)

In [ ]:
# Train the model
history = nn_model.fit(
    train_embeddings=train_embeddings,
    train_features=train_features,
    train_prices=train_prices,
    val_embeddings=val_embeddings,
    val_features=val_features,
    val_prices=val_prices,
    epochs=50,
    batch_size=64,
    verbose=True
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))
plt.plot(history['train_loss'], label='Train Loss', linewidth=2)
plt.plot(history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Neural Network Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate Neural Network
print("\nEvaluating Neural Network on test set...")
nn_predictions = nn_model.predict(test_embeddings, test_features)

nn_metrics = compute_metrics(nn_predictions, test_prices)

print("\n" + "="*60)
print("NEURAL NETWORK RESULTS")
print("="*60)
print(f"MAE: ${nn_metrics['mae']:.2f}")
print(f"RMSE: ${nn_metrics['rmse']:.2f}")
print(f"MAPE: {nn_metrics['mape']:.2f}%")
print(f"R²: {nn_metrics['r2']:.4f}")
print(f"Within 10%: {nn_metrics['within_10_pct']:.1f}%")
print(f"Within 20%: {nn_metrics['within_20_pct']:.1f}%")
print("="*60)

In [ ]:
# Save Neural Network model
nn_model.save('../data/models/neural_net_model.pth')

## Step 5: Compare All Models

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': ['RAG Baseline', 'Enhanced XGBoost', 'Neural Network'],
    'MAE': [
        compute_metrics([item.rag_mean_price for item in test], test_prices)['mae'],
        xgb_metrics['mae'],
        nn_metrics['mae']
    ],
    'RMSE': [
        compute_metrics([item.rag_mean_price for item in test], test_prices)['rmse'],
        xgb_metrics['rmse'],
        nn_metrics['rmse']
    ],
    'R²': [
        compute_metrics([item.rag_mean_price for item in test], test_prices)['r2'],
        xgb_metrics['r2'],
        nn_metrics['r2']
    ]
})

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MAE comparison
axes[0].bar(comparison['Model'], comparison['MAE'], color=['coral', 'skyblue', 'lightgreen'])
axes[0].set_ylabel('MAE ($)')
axes[0].set_title('Mean Absolute Error')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(comparison['MAE']):
    axes[0].text(i, v, f'${v:.2f}', ha='center', va='bottom')

# RMSE comparison
axes[1].bar(comparison['Model'], comparison['RMSE'], color=['coral', 'skyblue', 'lightgreen'])
axes[1].set_ylabel('RMSE ($)')
axes[1].set_title('Root Mean Squared Error')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(comparison['RMSE']):
    axes[1].text(i, v, f'${v:.2f}', ha='center', va='bottom')

# R² comparison
axes[2].bar(comparison['Model'], comparison['R²'], color=['coral', 'skyblue', 'lightgreen'])
axes[2].set_ylabel('R² Score')
axes[2].set_title('R-squared')
axes[2].tick_params(axis='x', rotation=45)
for i, v in enumerate(comparison['R²']):
    axes[2].text(i, v, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Step 6: Prediction Analysis

In [ ]:
# Scatter plots: Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# XGBoost
axes[0].scatter(test_prices, xgb_predictions, alpha=0.5, s=20, color='skyblue')
axes[0].plot([0, test_prices.max()], [0, test_prices.max()], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'XGBoost: MAE=${xgb_metrics["mae"]:.2f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Neural Network
axes[1].scatter(test_prices, nn_predictions, alpha=0.5, s=20, color='lightgreen')
axes[1].plot([0, test_prices.max()], [0, test_prices.max()], 'r--', label='Perfect prediction')
axes[1].set_xlabel('Actual Price ($)')
axes[1].set_ylabel('Predicted Price ($)')
axes[1].set_title(f'Neural Network: MAE=${nn_metrics["mae"]:.2f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Error distribution
xgb_errors = np.abs(xgb_predictions - test_prices)
nn_errors = np.abs(nn_predictions - test_prices)

plt.figure(figsize=(12, 6))
plt.hist(xgb_errors, bins=50, alpha=0.6, label='XGBoost', color='skyblue', edgecolor='black')
plt.hist(nn_errors, bins=50, alpha=0.6, label='Neural Network', color='lightgreen', edgecolor='black')
plt.xlabel('Absolute Error ($)')
plt.ylabel('Frequency')
plt.title('Error Distribution Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: Sample Predictions

In [ ]:
# Show some example predictions
print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

for i in range(5):
    item = test[i]
    actual = item.price
    xgb_pred = xgb_predictions[i]
    nn_pred = nn_predictions[i]
    rag_pred = item.rag_mean_price
    
    print(f"\n{i+1}. {item.title[:60]}...")
    print(f"   Category: {item.category}")
    print(f"   Actual Price: ${actual:.2f}")
    print(f"   RAG Baseline: ${rag_pred:.2f} (error: ${abs(rag_pred-actual):.2f})")
    print(f"   XGBoost: ${xgb_pred:.2f} (error: ${abs(xgb_pred-actual):.2f})")
    print(f"   Neural Net: ${nn_pred:.2f} (error: ${abs(nn_pred-actual):.2f})")

## Summary

✅ Trained Enhanced XGBoost with RAG features  
✅ Trained Deep Neural Network with embeddings  
✅ Evaluated both models on test set  
✅ Compared with RAG baseline  
✅ Analyzed feature importance  
✅ Saved trained models  

**Key Findings:**
- Both models significantly outperform RAG-only baseline
- RAG features are among the most important for XGBoost
- Neural network benefits from semantic embeddings

**Next:** Notebook 04 - Fine-tune LLM and create ensemble